HAWKEYE — T-HGNN TRAINING (Kaggle notebook)
============================================
Trains a Heterogeneous Graph Transformer (HGT) on the same RBI NFPC Phase 2
dataset that the LightGBM blend trained on. Outputs per-account embeddings
that the HAWKEYE backend can fuse into the scoring pipeline as a third model.

INPUT  (Kaggle dataset attached): /kaggle/input/.../rbih-nfpc-phase-2/
OUTPUT (downloaded after run):    /kaggle/working/hawkeye_thgnn/
                                    ├── thgnn_model.pt              (~5 MB)
                                    ├── thgnn_embeddings.parquet    (~10 MB, 160k rows × 128 dims)
                                    ├── thgnn_metadata.json         (AUC/F1/threshold/dim)
                                    └── node_id_maps.json           (account_id ↔ node_idx)

Runtime on Kaggle P100 / T4: ~25-40 minutes total.

After download, place the four files in c:/Users/dhruv/Code/hawkeye-idea/artifacts/
The HAWKEYE backend's EmbeddingService loads them automatically on next boot
and adds the 128-dim embedding as features to the LightGBM blend.

REQUIREMENTS (Kaggle has all):
  pip install torch>=2.1 torch_geometric>=2.5 lightgbm pandas numpy scikit-learn pyarrow

In [ ]:
# Kaggle's default image doesn't ship torch_geometric; install it (and its CSR/cluster deps).
# Pinned to versions that work with PyTorch 2.x in Kaggle's environment.
!pip install -q torch_geometric==2.6.1


## CELL 1 — IMPORTS & DATASET DISCOVERY

In [ ]:
import os, sys, gc, json, time
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} | "
      f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Auto-discover dataset path recursively (handles competition + dataset mount layouts)
BASE = None
for p in sorted(glob('/kaggle/input/**/train_labels.parquet', recursive=True)):
    BASE = os.path.dirname(p)
    break
if BASE is None:
    raise FileNotFoundError("Couldn't find train_labels.parquet anywhere under /kaggle/input/. Attach the RBI NFPC Phase 2 dataset.")
print(f"Dataset: {BASE}")

OUT = '/kaggle/working/hawkeye_thgnn'
os.makedirs(OUT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## CELL 2 — LOAD LABELS + ACCOUNT METADATA

In [ ]:
print("\n[1/6] Loading labels + account metadata + LightGBM artifacts (for 105 input features)...")
t0 = time.time()

train_labels    = pd.read_parquet(f'{BASE}/train_labels.parquet')
test_accounts   = pd.read_parquet(f'{BASE}/test_accounts.parquet')
accounts        = pd.read_parquet(f'{BASE}/accounts.parquet')

# Discover the hawkeye_artifacts dataset (output of ml-idea.ipynb).
ARTIFACTS_IN = None
for p in sorted(glob('/kaggle/input/**/account_feature_matrix.parquet', recursive=True)):
    parent = os.path.dirname(p)
    if os.path.exists(f'{parent}/feature_config.json'):
        ARTIFACTS_IN = parent
        break
if ARTIFACTS_IN is None:
    raise FileNotFoundError(
        "Couldn't find account_feature_matrix.parquet + feature_config.json under /kaggle/input/.\n"
        "Attach the hawkeye_artifacts Kaggle dataset (output of ml-idea.ipynb).")
print(f"  Artifacts: {ARTIFACTS_IN}")

with open(f'{ARTIFACTS_IN}/feature_config.json') as f:
    cfg = json.load(f)
feat_clean = cfg['feat_clean']
print(f"  Will use {len(feat_clean)} feat_clean columns as account-node inputs (was 4 raw cols).")

target_ids = pd.concat(
    [train_labels[['account_id']], test_accounts[['account_id']]]
).drop_duplicates()
all_aids = target_ids['account_id'].values

acct_to_idx = {aid: i for i, aid in enumerate(all_aids)}
n_accts     = len(all_aids)

y_full = pd.merge(target_ids, train_labels, on='account_id', how='left')['is_mule'].values
y_train_mask = ~np.isnan(y_full)
y_train = y_full[y_train_mask].astype(int)

print(f"  Accounts: {n_accts:,} | Train labels: {y_train_mask.sum():,} | Mules: {y_train.sum():,}")

## CELL 3 — STREAM TRANSACTIONS → BUILD HETEROGENEOUS EDGES

In [ ]:
print("\n[2/6] Streaming transactions to build hetero edges (this is the slow part)...")
t1 = time.time()

from tqdm.auto import tqdm

txn_parts = sorted(glob(f'{BASE}/transactions/batch-*/part_*.parquet'))
print(f"  {len(txn_parts)} transaction parquet parts (~1.6s/part on Kaggle CPU = ~10-12 min total)")

parts = []
target_set = set(all_aids)

pbar = tqdm(
    enumerate(txn_parts),
    total=len(txn_parts),
    desc='  edges',
    unit='part',
    smoothing=0.05,        # reactive ETA
    mininterval=2.0,       # don't spam updates
)
for i, fp in pbar:
    df = pd.read_parquet(fp, columns=[
        'account_id', 'counterparty_id', 'amount',
        'txn_type', 'transaction_timestamp'])
    df = df[df['account_id'].isin(target_set)]
    if len(df) == 0:
        continue
    df['amt']     = df['amount'].abs()
    df['is_cred'] = (df['txn_type'] == 'C').astype(np.int8)
    ts = pd.to_datetime(df['transaction_timestamp'], format='ISO8601', errors='coerce')
    df['hr']  = ts.dt.hour.fillna(12).astype(np.int8)
    df['dw']  = ts.dt.dayofweek.fillna(0).astype(np.int8)
    df['ngt'] = ((df['hr'] < 6) | (df['hr'] >= 23)).astype(np.int8)
    df['wkd'] = (df['dw'] >= 5).astype(np.int8)

    edge = df.groupby(['account_id', 'counterparty_id'], sort=False).agg(
        ew      =('amt', 'sum'),
        ec      =('amt', 'count'),
        ec_cred =('is_cred', 'sum'),
        ngt_n   =('ngt', 'sum'),
        wkd_n   =('wkd', 'sum'),
    ).reset_index()
    parts.append(edge)
    del df, edge
    gc.collect()

    if (i + 1) % 40 == 0:
        big = pd.concat(parts, ignore_index=True)
        parts = [big.groupby(['account_id', 'counterparty_id'], sort=False).agg(
            ew=('ew', 'sum'), ec=('ec', 'sum'), ec_cred=('ec_cred', 'sum'),
            ngt_n=('ngt_n', 'sum'), wkd_n=('wkd_n', 'sum')
        ).reset_index()]
        del big
        gc.collect()
        pbar.set_postfix(edges=f"{len(parts[0]):,}")

pbar.close()

# Final concat
edges = pd.concat(parts, ignore_index=True).groupby(
    ['account_id', 'counterparty_id'], sort=False).agg(
    ew=('ew', 'sum'), ec=('ec', 'sum'), ec_cred=('ec_cred', 'sum'),
    ngt_n=('ngt_n', 'sum'), wkd_n=('wkd_n', 'sum')
).reset_index()
del parts; gc.collect()

# Edge features
edges['cred_rate'] = edges['ec_cred'] / edges['ec'].clip(lower=1)
edges['ngt_rate']  = edges['ngt_n']  / edges['ec'].clip(lower=1)
edges['wkd_rate']  = edges['wkd_n']  / edges['ec'].clip(lower=1)
edges['log_ew']    = np.log1p(edges['ew']).astype(np.float32)
edges['log_ec']    = np.log1p(edges['ec']).astype(np.float32)

print(f"\n  Edges built: {len(edges):,} unique (account, counterparty) pairs in {time.time()-t1:.0f}s")

cps = edges['counterparty_id'].unique()
cp_to_idx = {cp: i for i, cp in enumerate(cps)}
n_cps = len(cps)
print(f"  Counterparty nodes: {n_cps:,}")

## CELL 4 — BUILD HETERODATA OBJECT

In [ ]:
print("\n[3/6] Building PyG HeteroData (105-d account features + edge-aggregated CP features)...")

from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

data = HeteroData()

# --- Account features: 105 LightGBM feat_clean columns ---------------
afm = pd.read_parquet(
    f'{ARTIFACTS_IN}/account_feature_matrix.parquet',
    columns=['account_id'] + feat_clean,
)
afm['account_id'] = afm['account_id'].astype(str)
afm = afm.set_index('account_id').reindex([str(a) for a in all_aids]).fillna(0)
acc_feat = afm[feat_clean].astype(np.float32).values
acc_mean = acc_feat.mean(axis=0)
acc_std  = acc_feat.std(axis=0) + 1e-6
acc_feat = (acc_feat - acc_mean) / acc_std
data['account'].x = torch.tensor(acc_feat, dtype=torch.float32)
print(f"  Account node features: {data['account'].x.shape}  (was [{n_accts}, 4])")

# --- Counterparty features: degree, sum_ew, mean credit/night/weekend rate
# PLUS edge-attribute aggregates (since HGTConv can't consume edge_attr directly).
cp_agg = edges.groupby('counterparty_id').agg(
    deg=('account_id', 'count'),
    sum_ew=('ew', 'sum'),
    mean_ew=('ew', 'mean'),
    sum_ec=('ec', 'sum'),
    mean_cred=('cred_rate', 'mean'),
    mean_ngt=('ngt_rate', 'mean'),
    mean_wkd=('wkd_rate', 'mean'),
).reindex(cps).fillna(0)
cp_feat = np.column_stack([
    np.log1p(cp_agg['deg'].values),
    np.log1p(cp_agg['sum_ew'].values),
    np.log1p(cp_agg['mean_ew'].values),
    np.log1p(cp_agg['sum_ec'].values),
    cp_agg['mean_cred'].values,
    cp_agg['mean_ngt'].values,
    cp_agg['mean_wkd'].values,
]).astype(np.float32)
cp_feat = (cp_feat - cp_feat.mean(0)) / (cp_feat.std(0) + 1e-6)
data['counterparty'].x = torch.tensor(cp_feat, dtype=torch.float32)
print(f"  Counterparty node features: {data['counterparty'].x.shape}  (was [{n_cps}, 4])")

# --- Edge index (edge_attr unused by HGTConv, dropped to save memory) -----
src_idx = edges['account_id'].map(acct_to_idx).values
dst_idx = edges['counterparty_id'].map(cp_to_idx).values
data['account', 'transacted', 'counterparty'].edge_index = torch.tensor(
    np.stack([src_idx, dst_idx]), dtype=torch.long)
data['counterparty', 'rev_transacted', 'account'].edge_index = torch.tensor(
    np.stack([dst_idx, src_idx]), dtype=torch.long)

# Train/test masks on accounts
train_mask = torch.zeros(n_accts, dtype=torch.bool)
y_tensor   = torch.zeros(n_accts, dtype=torch.float32)
for i, label in enumerate(y_full):
    if not np.isnan(label):
        train_mask[i] = True
        y_tensor[i]   = float(label)
data['account'].y          = y_tensor
data['account'].train_mask = train_mask

print(f"  HeteroData ready: {data}")

## CELL 5 — HGT MODEL

In [ ]:
print("\n[4/6] Defining HGT model (2 layers, residual + layernorm)...")

class HGTEncoder(nn.Module):
    '''Heterogeneous Graph Transformer.
    Default: 2 layers / 4 heads / 64-dim hidden - fits CPU training in ~25 min.
    Two layers = 2-hop reach (account -> CP -> account peers), which is what
    catches "mule ring shares the same hub counterparty" patterns.
    '''
    def __init__(self, metadata, in_dims, embed_dim=64, n_layers=2, n_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.lin_in = nn.ModuleDict({
            ntype: nn.Linear(in_dims[ntype], embed_dim) for ntype in metadata[0]
        })
        self.convs = nn.ModuleList([
            HGTConv(embed_dim, embed_dim, metadata, heads=n_heads)
            for _ in range(n_layers)
        ])
        self.norm = nn.ModuleList([
            nn.LayerNorm(embed_dim) for _ in range(n_layers)
        ])

    def forward(self, x_dict, edge_index_dict):
        x_dict = {k: F.elu(self.lin_in[k](v)) for k, v in x_dict.items()}
        for conv, norm in zip(self.convs, self.norm):
            new_x = conv(x_dict, edge_index_dict)
            x_dict = {k: norm(F.elu(v) + x_dict[k]) for k, v in new_x.items()}
        return x_dict


class HGTClassifier(nn.Module):
    '''HGT encoder + 2-layer MLP head over the account embedding.'''
    def __init__(self, metadata, in_dims, embed_dim=64, dropout=0.3):
        super().__init__()
        self.encoder = HGTEncoder(metadata, in_dims, embed_dim, n_layers=2)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        z = self.encoder(x_dict, edge_index_dict)
        logit = self.head(z['account']).squeeze(-1)
        return logit, z['account']

EMBED_DIM = 64
in_dims = {
    'account':      data['account'].x.size(1),  # now 105
    'counterparty': data['counterparty'].x.size(1),  # now 7
}

## CELL 6 — TRAIN (105-feat input, 2-layer HGT, early-stop)

In [ ]:
print("\n[5/6] Training HGT (single 80/20 holdout, early-stopping on val AUC)...")
data = data.to(device)
import gc as _gc

EPOCHS = 200      # was 50 — give it room to converge with the new feature load
LR     = 5e-4     # was 1e-3 — lower because input dim is now 105 vs 4
PATIENCE = 25     # epochs without val improvement before stopping

train_idx_all = torch.where(data['account'].train_mask)[0].cpu().numpy()
y_train_np    = data['account'].y[train_idx_all].cpu().numpy().astype(int)

from sklearn.model_selection import train_test_split
tr_local, va_local = train_test_split(
    np.arange(len(train_idx_all)),
    test_size=0.2,
    stratify=y_train_np,
    random_state=SEED,
)
tr_idx = torch.tensor(train_idx_all[tr_local], device=device)
va_idx = torch.tensor(train_idx_all[va_local], device=device)

model = HGTClassifier(data.metadata(), in_dims, EMBED_DIM).to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

pos = int(y_train_np[tr_local].sum())
neg = int((y_train_np[tr_local] == 0).sum())
pos_weight = torch.tensor(neg / max(pos, 1), device=device, dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_auc, best_epoch, best_state = 0., 0, None
no_improve = 0
print(f"  params={sum(p.numel() for p in model.parameters()):,}  pos_weight={pos_weight.item():.1f}")

for epoch in range(EPOCHS):
    model.train()
    opt.zero_grad()
    logit, _ = model(data.x_dict, data.edge_index_dict)
    loss = loss_fn(logit[tr_idx], data['account'].y[tr_idx])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()

    if (epoch + 1) % 2 == 0:
        model.eval()
        with torch.no_grad():
            logit_eval, _ = model(data.x_dict, data.edge_index_dict)
            p = torch.sigmoid(logit_eval[va_idx]).cpu().numpy()
            a = roc_auc_score(y_train_np[va_local], p)
            improved = a > best_auc + 1e-4
            if improved:
                best_auc, best_epoch = a, epoch + 1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 2
            if (epoch + 1) % 10 == 0 or improved:
                print(f"  epoch {epoch+1:3d} | loss {loss.item():.4f} | "
                      f"lr {opt.param_groups[0]['lr']:.2e} | val AUC {a:.5f}"
                      f"{'  <-- best' if improved else ''}")
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        _gc.collect()
        if no_improve >= PATIENCE:
            print(f"  early stop: no val improvement for {PATIENCE} epochs")
            break

print(f"\n  Best val AUC: {best_auc:.5f} at epoch {best_epoch}")
if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

model.eval()
with torch.no_grad():
    logit_final, embed_final = model(data.x_dict, data.edge_index_dict)
    final_embeddings = embed_final.cpu().numpy()
    final_proba      = torch.sigmoid(logit_final).cpu().numpy()
    p_va = final_proba[train_idx_all[va_local]]

oof_auc = float(roc_auc_score(y_train_np[va_local], p_va))
prec_v, rec_v, pr_thr = precision_recall_curve(y_train_np[va_local], p_va)
pr_f1 = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-9)
best_idx_thr = int(np.argmax(pr_f1))
oof_threshold = float(pr_thr[best_idx_thr])
oof_f1 = float(pr_f1[best_idx_thr])
fold_metrics = [oof_auc]

print(f"\n  T-HGNN holdout: AUC={oof_auc:.5f} | F1={oof_f1:.5f} @ thr={oof_threshold:.4f}")
del best_state
if device.type == 'cuda':
    torch.cuda.empty_cache()
_gc.collect()

## CELL 7 — EXPORT (final model trained on ALL labeled data + embeddings)

In [ ]:
print(f"\n[6/6] Saving exports to {OUT}/...")
final_model = model  # already trained on holdout; reuse the trained model
best_iters  = best_epoch

# ── Save model state (PyTorch native, ~5 MB)
torch.save({
    'state_dict': final_model.state_dict(),
    'in_dims':    in_dims,
    'embed_dim':  EMBED_DIM,
    'metadata':   data.metadata(),
}, f'{OUT}/thgnn_model.pt')

# ── Save per-account embeddings (160k × 128, ~80 MB raw, ~10 MB parquet w/ compression)
emb_df = pd.DataFrame(final_embeddings, columns=[f'thgnn_e{i:03d}' for i in range(EMBED_DIM)])
emb_df.insert(0, 'account_id', all_aids)
emb_df['thgnn_proba'] = final_proba
emb_df.to_parquet(f'{OUT}/thgnn_embeddings.parquet',
                   compression='zstd', index=False)

# ── Save node-id maps
with open(f'{OUT}/node_id_maps.json', 'w') as f:
    json.dump({
        'account_to_idx':      {a: i for a, i in list(acct_to_idx.items())[:1000]},  # sample only
        'counterparty_to_idx': {c: i for c, i in list(cp_to_idx.items())[:1000]},
        'n_accounts':          n_accts,
        'n_counterparties':    n_cps,
        'note': 'Full maps are reconstructed at serve-time from the parquet account_id column',
    }, f, indent=2)

# ── Save metadata
metadata = {
    'version':      'HAWKEYE_THGNN_v1',
    'oof_auc':      float(oof_auc),
    'oof_f1':       float(oof_f1),
    'oof_threshold': oof_threshold,
    'fold_aucs':    [float(a) for a in fold_metrics],
    'embed_dim':    EMBED_DIM,
    'n_accounts':   n_accts,
    'n_counterparties': n_cps,
    'n_edges':      int(data['account', 'transacted', 'counterparty'].edge_index.size(1)),
    'epochs':       best_iters,
    'training_seconds': time.time() - t0,
    'device':       str(device),
}
with open(f'{OUT}/thgnn_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n  ✓ Exports saved to {OUT}/")
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{f}') / 1024 / 1024
    print(f"    {f:<35} {sz:>8.2f} MB")

print(f"\n  T-HGNN OOF AUC: {oof_auc:.5f}  |  F1: {oof_f1:.5f}")
print(f"  Total time: {(time.time()-t0)/60:.1f} min")
import shutil
zip_path = shutil.make_archive('/kaggle/working/hawkeye_thgnn', 'zip', OUT)
print(f"\n  Zip ready: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
print("  Download /kaggle/working/hawkeye_thgnn.zip - unzip - drop into artifacts/")